# CALIPSO Downloader (interactive notebook)

This notebook provides an interactive interface to build download tasks for CALIPSO (L0-L3) products and run the low-level bash downloader `download_file.sh`. It supports: direct URL lists, URL templates with date expansion, and granule filename lists.

Notes:
- Keep credentials out of the notebook. Use `~/.netrc` or environment variables `EARTHDATA_USERNAME` and `EARTHDATA_PASSWORD`.
- Raw files are kept as-is by default.


In [1]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "certifi"], check=True)

  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
Using cached certifi-2026.7.22-py3-none-any.whl (136 kB)


CompletedProcess(args=['/Users/michaelbenneh/Documents/Git/WAFE/wafe_world/bin/python', '-m', 'pip', 'install', '--upgrade', 'certifi'], returncode=0)

In [2]:
import subprocess
import os
import sys
from pathlib import Path
from datetime import datetime, timedelta
from typing import List

def load_urls_from_file(path: str) -> List[str]:
    with open(path, 'r') as fh:
        return [line.strip() for line in fh if line.strip() and not line.startswith('#')]

def daterange(start_date, end_date):
    for n in range(int((end_date - start_date).days) + 1):
        yield start_date + timedelta(n)

def expand_template(template: str, product: str = None, start: str = None, end: str = None, granules: List[str] = None) -> List[str]:
    urls = []
    if granules:
        if start and end:
            s = datetime.strptime(start, '%Y-%m-%d')
            e = datetime.strptime(end, '%Y-%m-%d')
            for dt in daterange(s, e):
                for g in granules:
                    url = template.format(PRODUCT=product or '', YYYY=dt.strftime('%Y'), MM=dt.strftime('%m'), DD=dt.strftime('%d'), YYYYMMDD=dt.strftime('%Y%m%d'), FNAME=g)
                    urls.append(url)
        else:
            for g in granules:
                url = template.format(PRODUCT=product or '', YYYY='', MM='', DD='', YYYYMMDD='', FNAME=g)
                urls.append(url)
    else:
        if not start or not end:
            raise ValueError('start and end required when not using granule-list')
        s = datetime.strptime(start, '%Y-%m-%d')
        e = datetime.strptime(end, '%Y-%m-%d')
        for dt in daterange(s, e):
            url = template.format(PRODUCT=product or '', YYYY=dt.strftime('%Y'), MM=dt.strftime('%m'), DD=dt.strftime('%d'), YYYYMMDD=dt.strftime('%Y%m%d'))
            urls.append(url)
    return urls


In [3]:
def build_download_cmd(url: str, outdir: str, netrc_file: str = None, username: str = None, password: str = None) -> List[str]:
    script = Path('download_file.sh').resolve()  # assume notebook runs in obs_downloader/ or adjust path accordingly
    cmd = [str(script), url, outdir]
    if netrc_file:
        cmd += ['--netrc-file', netrc_file]
    if username and password:
        cmd += ['--username', username, '--password', password]
    return cmd

def run_download(url: str, outdir: str, netrc_file: str = None, username: str = None, password: str = None) -> subprocess.CompletedProcess:
    cmd = build_download_cmd(url, outdir, netrc_file=netrc_file, username=username, password=password)
    print('Running:', ' '.join(cmd))
    return subprocess.run(cmd, text=True)

def download_urls(urls: List[str], outdir: str, concurrency: int = 1, netrc_file: str = None, username: str = None, password: str = None, dry_run: bool = False):
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    if dry_run:
        for u in urls:
            print('DRY:', ' '.join(build_download_cmd(u, str(outdir), netrc_file=netrc_file, username=username, password=password)))
        return

    results = []
    for index, u in enumerate(urls, start=1):
        print(f'Downloading {index}/{len(urls)}: {u}')
        try:
            res = run_download(u, str(outdir), netrc_file, username, password)
            if res.returncode == 0:
                print(f'DONE {index}/{len(urls)}: {u}')
            else:
                print(f'FAILED {index}/{len(urls)}: {u} -> returncode={res.returncode}')
            results.append((u, res.returncode))
        except Exception as e:
            print('EXC for', u, e)
            results.append((u, -1))
    return results


In [6]:
# Automatic CMR virtual-directory download: change only these inputs.
PROJECT_DIR = Path.cwd()  # Or use a string such as '/path/to/Ba_Sing_Se'.
PROJECT_DIR = Path(PROJECT_DIR).expanduser().resolve()
SOURCE = 'asdc_calipso_l0_virtual_directory'
START = '2006-07-02'
END = '2006-07-02'
OUTDIR = 'downloads/calipso-l0'
CONCURRENCY = 1
NETRC_FILE = None  # Use ~/.netrc for protected CALIPSO L0 downloads.
DRY_RUN = False  # Change to False to download files.

# No URL template or filename is needed: the selected CMR profile builds the
# YYYY/MM/DD virtual-directory URL and discovers the HDF files automatically.
SCRIPT = PROJECT_DIR / 'obs_download.py'
CONFIG = PROJECT_DIR / 'config_template.yaml'
command = [sys.executable, str(SCRIPT), '--config', str(CONFIG), '--source', SOURCE,
           '--start', START, '--end', END, '--outdir', OUTDIR,
           '--concurrency', str(CONCURRENCY)]
if NETRC_FILE:
    netrc_path = Path.home() / '.netrc' if NETRC_FILE is True else Path(NETRC_FILE).expanduser()
    command += ['--netrc-file', str(netrc_path)]
if DRY_RUN:
    command.append('--dry-run')

print('Running:', ' '.join(command))
result = subprocess.run(command, text=True)
if result.returncode:
    raise RuntimeError(f'Downloader exited with status {result.returncode}')


Running: /Users/michaelbenneh/Documents/Git/WAFE/wafe_world/bin/python /Users/michaelbenneh/Documents/Git/WAFE/Ba_Sing_Se/obs_download.py --config /Users/michaelbenneh/Documents/Git/WAFE/Ba_Sing_Se/config_template.yaml --source asdc_calipso_l0_virtual_directory --start 2006-07-02 --end 2006-07-02 --outdir downloads/calipso-l0 --concurrency 1
Querying CMR virtual directory: https://cmr.earthdata.nasa.gov/virtual-directory/collections/C3880519029-LARC_CLOUD/temporal/2006/07/02
Resolved 16 file(s).


curl: (22) The requested URL returned error: 401                               

FAILED 1/16: https://data.asdc.earthdata.nasa.gov/asdc-prod-protected/CALIPSO/CAL_LID_L0-Standard-V1-00_V1-00/2006.07/CAL_LID_L0-Standard-V1-00.2006-07-02T00-00-00Z.hdf -> returncode=22


curl: (22) The requested URL returned error: 401                               

FAILED 2/16: https://data.asdc.earthdata.nasa.gov/asdc-prod-protected/CALIPSO/CAL_LID_L0-Standard-V1-00_V1-00/2006.07/CAL_LID_L0-Standard-V1-00.2006-07-02T01-30-00Z.hdf -> returncode=22


curl: (22) The requested URL returned error: 401                               

FAILED 3/16: https://data.asdc.earthdata.nasa.gov/asdc-prod-protected/CALIPSO/CAL_LID_L0-Standard-V1-00_V1-00/2006.07/CAL_LID_L0-Standard-V1-00.2006-07-02T03-00-00Z.hdf -> returncode=22


curl: (22) The requested URL returned error: 401                               

FAILED 4/16: https://data.asdc.earthdata.nasa.gov/asdc-prod-protected/CALIPSO/CAL_LID_L0-Standard-V1-00_V1-00/2006.07/CAL_LID_L0-Standard-V1-00.2006-07-02T04-30-00Z.hdf -> returncode=22


...

curl: (22) The requested URL returned error: 401                               

FAILED 13/16: https://data.asdc.earthdata.nasa.gov/asdc-prod-protected/CALIPSO/CAL_LID_L0-Standard-V1-00_V1-00/2006.07/CAL_LID_L0-Standard-V1-00.2006-07-02T18-00-00Z.hdf -> returncode=22


curl: (22) The requested URL returned error: 401                               

FAILED 14/16: https://data.asdc.earthdata.nasa.gov/asdc-prod-protected/CALIPSO/CAL_LID_L0-Standard-V1-00_V1-00/2006.07/CAL_LID_L0-Standard-V1-00.2006-07-02T19-30-00Z.hdf -> returncode=22


curl: (22) The requested URL returned error: 401                               

FAILED 15/16: https://data.asdc.earthdata.nasa.gov/asdc-prod-protected/CALIPSO/CAL_LID_L0-Standard-V1-00_V1-00/2006.07/CAL_LID_L0-Standard-V1-00.2006-07-02T21-00-00Z.hdf -> returncode=22


curl: (22) The requested URL returned error: 401                               

FAILED 16/16: https://data.asdc.earthdata.nasa.gov/asdc-prod-protected/CALIPSO/CAL_LID_L0-Standard-V1-00_V1-00/2006.07/CAL_LID_L0-Standard-V1-00.2006-07-02T22-30-00Z.hdf -> returncode=22


RuntimeError: Downloader exited with status 1